# M0 GPU checks — prefix-fork on Qwen3.5-0.8B-Base

Runs the fork equivalence tests with the CUDA kernels (`flash-linear-attention`, `causal-conv1d`) and benchmarks fork vs. one full forward per branch.

**Runtime → Change runtime type → GPU** (T4 is fine), then **Runtime → Run all**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
%cd /content
!rm -rf qwen-rlcd && git clone -q -b worktree-m0-gpu-checks https://github.com/shamazharikh/qwen-rlcd
%cd /content/qwen-rlcd
!git log --oneline -1
!pip install -q -e '.[dev]'

## Install CUDA kernels
`causal-conv1d` downloads a prebuilt wheel when one matches this torch/CUDA/Python; otherwise it compiles (can take 10+ minutes).

In [ ]:
!pip install -q flash-linear-attention
!pip install -q causal-conv1d --no-build-isolation
!python -c "import torch, transformers, fla, causal_conv1d; print('torch', torch.__version__, '| transformers', transformers.__version__, '| fla', fla.__version__, '| causal_conv1d', causal_conv1d.__version__)"

## Fork equivalence tests on CUDA (tiny model + real weights; fp32, plus bf16 on Ampere+)

In [ ]:
!QWEN_RLCD_SLOW=1 python -m pytest -q -s 2>&1 | tee test.log | grep -vE 'Loading weights|Warning: You are sending'
!echo; grep -q 'falling back' test.log && (echo 'WARNING: torch fallback kernels were used:'; grep 'falling back' test.log | sort -u) || echo 'OK: no torch fallbacks, CUDA kernels in use'

## Benchmark: fork vs. sequential

In [ ]:
!python scripts/bench_fork.py 2>&1 | grep -vE 'Loading weights|Warning: You are sending|falling back'